# Тема 5. Метод опорных векторов и работа с признаками

Студенческий notebook. Заполните блоки **TODO**. Не используйте test до финального раздела.

## 0. Настройка окружения

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     cross_validate, GridSearchCV, cross_val_predict)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.feature_selection import (SelectKBest, f_classif, mutual_info_classif,
                                       RFECV, SelectFromModel)
from sklearn.decomposition import PCA
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             classification_report, confusion_matrix,
                             RocCurveDisplay, PrecisionRecallDisplay)

RANDOM_STATE = 42
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
np.random.seed(RANDOM_STATE)

## 1. Данные и data card
Target в sklearn: `0 = malignant`, `1 = benign`. При расчёте recall malignant учитывайте направление положительного класса.

In [ ]:
data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
print(X.shape)
print(pd.Series(y).value_counts().rename(index=dict(enumerate(data.target_names))))
X.head()

In [ ]:
# TODO: проверить типы, пропуски, дубликаты, диапазоны и корреляции
# Ваш код


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)
print(X_train.shape, X_test.shape)

## 2. Baseline

In [ ]:
scoring = {'roc_auc':'roc_auc', 'ap':'average_precision', 'accuracy':'accuracy'}
models = {
    'dummy': DummyClassifier(strategy='prior'),
    'logreg': Pipeline([('scale', StandardScaler()),
                        ('model', LogisticRegression(max_iter=5000))])
}
# TODO: cross_validate обе модели и собрать mean/std в DataFrame


## 3. Масштабирование

In [ ]:
scalers = {
    'none': 'passthrough',
    'standard': StandardScaler(),
    'robust': RobustScaler(),
    'minmax': MinMaxScaler()
}
# TODO: для каждого scaler оценить SVC(C=1, gamma='scale') на одинаковых folds
# Сравнить ROC-AUC, std и fit_time


## 4. LinearSVC и C

In [ ]:
c_grid = np.logspace(-4, 3, 8)
# TODO: построить validation curve вручную через cross_validate
# Для каждого C сохранить mean_train_score и mean_test_score


In [ ]:
# TODO: обучить лучший LinearSVC на train, получить коэффициенты
# Визуализировать top-10 по abs(coef_); обсудить коррелированные признаки


## 5. RBF SVM: совместный поиск C и gamma

In [ ]:
rbf_pipe = Pipeline([
    ('scale', StandardScaler()),
    ('model', SVC(kernel='rbf', class_weight='balanced'))
])
param_grid = {
    'model__C': np.logspace(-3, 3, 7),
    'model__gamma': np.logspace(-4, 1, 6)
}
search = GridSearchCV(rbf_pipe, param_grid, scoring='roc_auc', cv=cv,
                      n_jobs=-1, return_train_score=True)
# TODO: fit только на X_train, y_train


In [ ]:
# TODO: преобразовать cv_results_ в pivot table и построить heatmap
# 1) mean_test_score; 2) mean_train_score - mean_test_score


In [ ]:
# TODO: вывести best_params_, best_score_ и n_support_ лучшей модели


## 6. Сравнение ядер

In [ ]:
# TODO: задать отдельные param_grid для linear, poly и rbf
# Использовать один cv, одну scoring и сравнить качество + fit_time


## 7. Filter selection внутри Pipeline

In [ ]:
filter_pipe = Pipeline([
    ('scale', StandardScaler()),
    ('select', SelectKBest()),
    ('model', SVC(kernel='rbf'))
])
filter_grid = [
    {'select__score_func':[f_classif], 'select__k':[5,10,15,20,25,'all'],
     'model__C':[0.1,1,10], 'model__gamma':['scale',0.01,0.1]},
    {'select__score_func':[mutual_info_classif], 'select__k':[5,10,15,20,25,'all'],
     'model__C':[0.1,1,10], 'model__gamma':['scale',0.01,0.1]}
]
# TODO: GridSearchCV; сравнить score_func и k


## 8. Embedded selection и RFECV

In [ ]:
embedded = Pipeline([
    ('scale', StandardScaler()),
    ('select', SelectFromModel(
        LinearSVC(penalty='l1', dual=False, C=0.05, max_iter=10000)
    )),
    ('model', SVC(kernel='rbf', C=1, gamma='scale'))
])
# TODO: CV-оценка embedded; извлечь mask выбранных признаков после fit на train


In [ ]:
# TODO: RFECV с линейным estimator. Учтите вычислительную стоимость.


## 9. PCA: extraction, а не selection

In [ ]:
# TODO: сравнить Pipeline(StandardScaler, PCA(n_components=v), SVC)
# для v = 0.90, 0.95, 0.99 и baseline без PCA


## 10. Out-of-fold порог

In [ ]:
# Внимание: в sklearn class 1 = benign. Для malignant удобно инвертировать score.
best_model = None  # TODO: зафиксировать итоговый estimator до test
# oof_score_benign = cross_val_predict(best_model, X_train, y_train,
#                                      cv=cv, method='decision_function', n_jobs=-1)
# score_malignant = -oof_score_benign
# y_malignant = (y_train == 0).astype(int)
# TODO: выбрать threshold с recall >= 0.95 и максимальной precision


## 11. Финальная оценка на test — открыть только сейчас

In [ ]:
# TODO: fit итоговой модели на всём train
# TODO: один раз получить test scores и применить зафиксированный threshold
# ROC-AUC, AP, classification_report, confusion_matrix, ROC/PR curves


## 12. Noise features и устойчивость

In [ ]:
# Создаём шум отдельно после split, не используя target
rng = np.random.default_rng(RANDOM_STATE)
X_train_noise = X_train.copy()
X_test_noise = X_test.copy()
for j in range(20):
    X_train_noise[f'noise_{j}'] = rng.normal(size=len(X_train_noise))
    X_test_noise[f'noise_{j}'] = rng.normal(size=len(X_test_noise))
# TODO: повторить selection внутри CV; сколько noise features выбрано?


## 13. Итоговый memo

Заполните: **модель**, **преобразования**, **C/gamma**, **CV-оценка**, **test-оценка**, **порог**, **стоимость ошибок**, **ограничения**, **мониторинг drift**.